# Plots for company retreat 2026

In [ ]:
from raman_analysis.utils import collect_spectra_from_dir, plot_spectra_in_dir, add_peak_annotations
import plotly.express as px
import ramanspy as rp
import pandas as pd
import plotly.colors as pcolors
import numpy as np
from pathlib import Path
import arcadia_pycolor as apc
apc.plotly.setup()
%load_ext autoreload
%autoreload 2

real_NAS = Path('/Volumes/Microscopy')
NAS_mirror = Path('/Users/ilya/NAS_mirror')
pub_data_dir = Path('data/')
plot_dir_htmls = Path('plots/html')
plot_dir_pdf = Path('plots/pdf')

plot_scale = 3
fig_width = 1000
fig_height=400

In [ ]:
apc.palettes.primary

In [ ]:
def generate_subtract_dict(samples):
    """ Generate a dictionary mapping sample names to their buffer names for subtraction. """
    subtract_dict = {}
    for name in samples:
        date = name.split('/')[0]
        buffer = name.split(' in ')[-1]
        subtract_dict[name] = f"{date}/{buffer}"
    return subtract_dict

# Playground (will delete)

In [ ]:
# data_dir = pub_data_dir

# data_list, sample_df = collect_spectra_from_dir(data_dir, 'instantraman',recursive=True)

# pipeline = rp.preprocessing.Pipeline(
#     [
#         rp.preprocessing.misc.Cropper(region=(350, 2000)),
#         # rp.preprocessing.despike.WhitakerHayes(),
#         rp.preprocessing.denoise.SavGol(window_length=5, polyorder=3),
#         # rp.preprocessing.baseline.ASPLS(lam=1e5),
#         rp.preprocessing.baseline.ModPoly(poly_order=4),
#         # rp.preprocessing.normalise.AUC(),
#     ]
# )

# n_groups = sample_df['Group ID'].nunique()

# include_samples = [n for n in sample_df['Name'] if "Hemoglobin [low]" in n]
# # include_samples = [n for n in sample_df['Name'] if any(x in n for x in [" S4", " S5"])]
# # include_samples = [n for n in sample_df['Name'] if any(x in n for x in [" S1", " S2", " S3"])]

# sampled_custom_colors = pcolors.sample_colorscale(px.colors.qualitative.Dark2, samplepoints=n_groups)

# subtract_id = generate_subtract_dict(include_samples)

# f, names, rp_container = plot_spectra_in_dir(data_list,
#                     col_to_plot = "Intensity (dark-subtracted)",
#                     data_dir = data_dir,
#                     include=None,
#                     rp_pipeline=pipeline,
#                     colors=sampled_custom_colors,
#                     height=600,
#                     width=1000,
#                     subtract_id=subtract_id,# '2025-12-10/Tris',
#                     # y_offset=0.002,
#                     xaxis_title='Wavenumber (cm⁻¹)',
#                     yaxis_title='Intensity (a.u.)',
# )
# # f.write_html("plots/2025-10-31-buffers.html")
# f.show()


# Protein perturbations

* Data from 2025-12-{10,13}

In [ ]:
data_dir = pub_data_dir

data_list, sample_df = collect_spectra_from_dir(data_dir, 'instantraman', recursive=True)

pipeline = rp.preprocessing.Pipeline(
    [
        rp.preprocessing.misc.Cropper(region=(350, 1800)),
        rp.preprocessing.despike.WhitakerHayes(),
        rp.preprocessing.denoise.SavGol(window_length=5, polyorder=3),
        # rp.preprocessing.baseline.ASPLS(lam=5e4),
        rp.preprocessing.baseline.ModPoly(poly_order=9),
        # rp.preprocessing.normalise.AUC(),
    ]
)

xranges = {'BSA(10mgml)': [[None,1500], [360, 700], [1150,1378]],
           'DNAse': [None, [360, 700], [1150,1378]],
           'CytC [high]': [None, [360, 700], [1500,1650]]
           }
yranges = {'BSA(10mgml)': [None, [-700,1500], None],
           'DNAse': [None, [-300, 1500], None],
           'CytC [high]': [None, None, None]}

fig_heights = {'BSA(10mgml)': [fig_height, fig_height, fig_height],
               'DNAse': [fig_height, fig_height, fig_height],
               'CytC [high]': [fig_height, fig_height, fig_height]}
fig_widths = {'BSA(10mgml)': [fig_width, fig_width*0.5, fig_width*0.5],
              'DNAse': [fig_width, fig_width*0.5, fig_width*0.5],
              'CytC [high]': [fig_width, fig_width*0.5, fig_width*0.5]}

n_groups = sample_df['Group ID'].nunique()

sampled_custom_colors = apc.palettes.primary # pcolors.sample_colorscale(px.colors.qualitative.Bold, samplepoints=n_groups)


buffers = {'BSA(10mgml)': ['Tris', ' Tris+TCEP', ' Tris+EDTA'],
           'DNAse': ['Tris', 'Tris+TCEP', 'Tris+EDTA', 'Tris+TCEP+EDTA'],
           'CytC [high]': ['Tris', 'Tris+TCEP', 'Tris+EDTA', 'Tris+TCEP+EDTA']}


for protein in xranges.keys():
    for i, (xrange, yrange) in enumerate(zip(xranges[protein], yranges[protein])):
        include_samples = [n for n in sample_df['Name'] if (any(n.endswith(buf) for buf in buffers[protein]) and protein in n)]
        subtract_id = generate_subtract_dict(include_samples)
        include_samples += list(subtract_id.values())
        f, names, rp_container = plot_spectra_in_dir(data_list,
                            col_to_plot = "Intensity (dark-subtracted)",
                            data_dir = data_dir,
                            subtract_id=subtract_id, # None if ('BSA' in protein) and ('Tris+TCEP+EDTA' in buffer_subset) else subtract_id,
                            rp_pipeline=pipeline,
                            colors=sampled_custom_colors,
                            include=include_samples,
                            height=fig_heights[protein][i],
                            width=fig_widths[protein][i],
                            # y_offset=0.002,
                            xaxis_title='Wavenumber (cm⁻¹)',
                            yaxis_title='Intensity (a.u.)',
                            title=f"{protein} | {xrange}_cm-1 | {'_'.join(buffers[protein])}"
                            )
        f.update_xaxes(range=xrange)
        f.update_yaxes(range=yrange)
        # f.data = tuple(trace for trace in f.data if trace.name != subtract_id)
        f.data = tuple(trace for trace in f.data if trace.name not in subtract_id.values())
        f.update_layout(showlegend=False)
        apc.plotly.style_plot(f, monospaced_axes="all")

        
        protein_dir_pdf = plot_dir_pdf / protein
        protein_dir_html = plot_dir_htmls / protein
        protein_dir_pdf.mkdir(parents=True, exist_ok=True)
        protein_dir_html.mkdir(parents=True, exist_ok=True)
        
        f.write_image(protein_dir_pdf / f"{protein}_{xrange}_cm-1.pdf", scale=plot_scale)
        f.write_html(protein_dir_html / f"{protein}_{xrange}_cm-1.html")
        f.show()



## Buffers alone

In [ ]:
buffers = ['Tris', 'Tris+TCEP', 'Tris+EDTA', 'Tris+TCEP+EDTA']
include_samples = [n for n in sample_df['Name'] if (any(n == f'2025-12-10/{buf}' for buf in buffers))]

f, names, rp_container = plot_spectra_in_dir(data_list,
                    col_to_plot = "Intensity (dark-subtracted)",
                    data_dir = data_dir,
                    subtract_id=None,
                    rp_pipeline=pipeline,
                    colors=sampled_custom_colors,
                    include=include_samples,
                    height=fig_height,
                    width=fig_width,
                    # y_offset=0.002,
                    xaxis_title='Wavenumber (cm⁻¹)',
                    yaxis_title='Intensity (a.u.)',
                    title="Buffers",
                    )

f.update_layout(showlegend=False)
apc.plotly.style_plot(f, monospaced_axes="all")
f.write_image(plot_dir_pdf / "buffers.pdf", scale=plot_scale)
f.write_html(plot_dir_htmls / "buffers.html")
f.show()